# 📈 Backtest Report – Quant Analyst
**Workflow**: Signal Prep → Vectorized Backtest → Monte Carlo Simulation → Monthly PnL Analytics
---

In [13]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

session = AnalystSession(role=RoleContext.ANALYST)
SYMBOL = "BTC-USD"
TIMEFRAME = "1d"
TRANSACTION_COST = 0.0005
INITIAL_CAPITAL = 100000

PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

## 1. Strategy Definition
Generating internal signals for backtesting.

In [14]:
try:
    df = session.load_ohlcv(SYMBOL, TIMEFRAME, days=730)
except Exception:
    df = session.sample_ohlcv(symbol="BTC", days=365)

df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 20])

# Basic Trend-Following Cross: SMA(21) vs SMA(50)
df = df.with_columns(
    pl.when(pl.col('sma_5') > pl.col('sma_20')).then(1).otherwise(-1).alias('signal')
)
df.head()

timestamp,open,high,low,close,volume,returns,sma_5,vol_5,sma_20,vol_20,avg_gain,avg_loss,rsi_14,signal
"datetime[μs, Asia/Ho_Chi_Minh]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
2024-03-17 07:00:00 +07,65254.22,68877.82,64505.0,68343.64,12631.850841,null,null,null,null,null,null,null,null,-1
2024-03-18 07:00:00 +07,68354.29,68933.71,66562.65,67613.04,20057.3758,-0.01069,null,null,null,null,null,null,null,-1
2024-03-19 07:00:00 +07,67609.39,68136.39,61506.0,61906.27,40111.97554,-0.084403,null,null,null,null,null,null,null,-1
2024-03-20 07:00:00 +07,61902.57,68150.0,60771.14,67858.8,38590.783455,0.096154,null,null,null,null,null,null,null,-1
2024-03-21 07:00:00 +07,67855.88,68234.56,64525.0,65484.7,18523.962472,-0.034986,66241.29,2659.075921,null,null,null,null,null,-1


## 2. Interactive Performance Overview
Strategy Equity Curve vs Buy & Hold Benchmark.

In [15]:
pf = session.run_vbt_backtest(df, signal_col='signal', transaction_cost=TRANSACTION_COST)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['timestamp'], y=pl.Series("equity_curve", pf.value()), name='Strategy', line=dict(color='#34d399')))
bh = (1 + df['returns'].fill_null(0)).cum_prod()
fig.add_trace(go.Scatter(x=df['timestamp'], y=bh, name='Buy & Hold', line=dict(color='#64748b', dash='dash')))

fig.update_layout(
    title=f"{SYMBOL} Strategy Cumulative Performance",
    yaxis_title="Relative Equity",
    **PLOTLY_DARK
)
fig.show()

## 3. Monte Carlo Robustness Test
Shuffling daily returns 250 times to test if the strategy's performance is luck-based.

In [16]:
n_sims = 250
strat_returns = pl.Series("net_return", pf.returns()).drop_nulls().to_numpy()
sim_results = []

for _ in range(n_sims):
    shuffled = np.random.choice(strat_returns, size=len(strat_returns), replace=True)
    sim_results.append(np.cumprod(1 + shuffled))

fig_mc = go.Figure()
for sim in sim_results[:50]: # plot subset for speed
    fig_mc.add_trace(go.Scatter(y=sim, mode='lines', line=dict(width=0.5), opacity=0.3, showlegend=False, line_color='#94a3b8'))
    
fig_mc.add_trace(go.Scatter(y=np.percentile(sim_results, 50, axis=0), name='Median Sim', line=dict(color='#fef08a', width=2)))
fig_mc.add_trace(go.Scatter(y=np.cumprod(1 + strat_returns), name='Actual Strategy', line=dict(color='#34d399', width=3)))

fig_mc.update_layout(
    title=f"Monte Carlo Simulation ({n_sims} paths)",
    xaxis_title="Days",
    yaxis_title="Equity",
    **PLOTLY_DARK
)
fig_mc.show()

## 4. Advanced Risk Metrics
Computing drawdown profiles and VaR.

In [17]:
metrics = session.compute_extended_metrics(pl.Series("equity_curve", pf.value()))
metrics_df = pl.DataFrame({"Metric": list(metrics.keys()), "Value": list(metrics.values())})
display(metrics_df)

equity = pl.Series("equity_curve", pf.value()).to_numpy()
peaks = np.maximum.accumulate(equity)
drawdowns = (equity - peaks) / peaks

fig_dd = px.area(x=df['timestamp'], y=drawdowns, title="Drawdown Profile", labels={'y': 'Drawdown %'})
fig_dd.update_traces(fillcolor='#f87171', line_color='#ef4444')
fig_dd.update_layout(**PLOTLY_DARK)
fig_dd.show()

Metric,Value
str,f64
"""total_return""",0.136834
"""annualized_return""",0.072844
"""annualized_vol""",0.239845
"""sharpe_ratio""",0.303714
"""max_drawdown""",-0.29541
…,…
"""calmar_ratio""",0.2466
"""win_rate""",0.2442
"""profit_factor""",1.081


## 5. Monthly Return Heatmap
Visualizing seasonality and periodicity in returns.

In [18]:
# Robust monthly heatmap using AnalystSession helper
monthly_ret = session.get_monthly_returns(bt)
heatmap_data = monthly_ret.select(pl.all().exclude(['year', 'YTD']))

fig_heat = px.imshow(
    heatmap_data.to_numpy(),
    x=heatmap_data.columns,         # Jan...Dec
    y=monthly_ret['year'].to_numpy(),
    color_continuous_scale='RdYlGn',
    origin='upper',
    title="Monthly Return Heatmap (%)"
)
fig_heat.update_layout(**PLOTLY_DARK)
fig_heat.show()

NameError: name 'bt' is not defined